# Rain Prediction — Imputation / Data-Quality Audit

**Decision notebook.** EXP-2026-06-15-002 concluded the operating-point F1 ceiling (~0.66) is
*data-limited*, with a k-NN Bayes-error proxy of ~0.16 measured in the **same-day, MICE-imputed**
feature space. But the raw WeatherAUS data is **48% / 43% / 41% / 38% missing** on
`Sunshine` / `Evaporation` / `Cloud3pm` / `Cloud9am` — several of which are top model predictors.
So a large share of the signal is *reconstructed*, not observed.

**Question:** how much of the ceiling is **imputation noise** (→ better data/imputation pays off,
keep missingness features) vs **genuine meteorological ambiguity** (→ stop digging here; pivot to
temporal / spatial features or a different target)?

Three cheap tests answer it:
- **A.** Do `*_was_imputed` indicator features improve F1? (Is missingness itself signal?)
- **B.** Does predictability degrade monotonically with the number of imputed key features? (Dose-response.)
- **C.** Is the noise floor lower on fully-observed rows than on the full set?

Outputs are mirrored to `AUDIT_result.txt`; figures saved under `figures/` as `audit_*`.
Runs on Google Colab (GPU). See [`XGB_experiments.md`](./XGB_experiments.md) for the predecessor experiments.


## 1. Colab setup, imports and constants

In [ ]:
# --- Colab environment setup ---
import subprocess, sys as _sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([_sys.executable, '-m', 'pip', '-q', 'install', '-U', 'xgboost'], check=False)
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)
    # To read the CSVs from Google Drive, uncomment:
    # from google.colab import drive
    # drive.mount('/content/drive')

print(f'Running on Colab: {IN_COLAB}')

In [ ]:
import os
import sys
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score, roc_auc_score,
)

import xgboost as xgb
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')

# --- GPU configuration ---
def _gpu_available():
    try:
        dtest = xgb.DMatrix(np.zeros((8, 2)), label=np.zeros(8))
        xgb.train({'device': 'cuda', 'tree_method': 'hist'}, dtest, num_boost_round=1)
        return True
    except Exception as e:
        print(f'CUDA probe failed ({type(e).__name__}); falling back to CPU.')
        return False

USE_GPU = _gpu_available()
DEVICE = 'cuda' if USE_GPU else 'cpu'
SEARCH_N_JOBS = 1 if USE_GPU else -1

XGB_KW = dict(tree_method='hist', device=DEVICE, eval_metric='logloss',
              random_state=RANDOM_STATE, n_jobs=-1)

def make_xgb(**overrides):
    params = dict(XGB_KW); params.update(overrides)
    return XGBClassifier(**params)

# Frozen deployment-candidate config from EXP-2026-06-15-002.
V2_PARAMS = dict(
    max_depth=6, min_child_weight=10, gamma=2.0, reg_lambda=20, reg_alpha=0.5,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_estimators=800,
)
AUDIT_THRESHOLD = 0.65   # v2 PR-crossover operating point

print(f'xgboost {xgb.__version__} | device: {DEVICE}')

### Output setup

In [ ]:
os.makedirs('figures', exist_ok=True)
_RESULT_PATH = 'AUDIT_result.txt'
_RESULT_FILE = open(_RESULT_PATH, 'w', encoding='utf-8')

class _Tee:
    def __init__(self, *s): self._s = s
    def write(self, d):
        for s in self._s: s.write(d)
    def flush(self):
        for s in self._s: s.flush()
    def isatty(self): return False

_ORIG_STDOUT = sys.stdout
sys.stdout = _Tee(_ORIG_STDOUT, _RESULT_FILE)
print('# Imputation / data-quality audit — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def log_section(title):
    bar = '=' * 72
    print(f'\n{bar}\n{title}\n{bar}')

def save_fig(name, fig=None):
    fig = fig or plt.gcf()
    path = f'figures/{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  [saved {path}]')

## 2. Load raw (pre-MICE) + preprocessed (post-MICE)

In [ ]:
log_section('2. Data loading')

def discover(cands):
    return next((p for p in cands if os.path.exists(p)), None)

RAW_PATH = discover([
    'weatherAUS.csv', 'data/weatherAUS.csv', '../data/weatherAUS.csv',
    '/content/weatherAUS.csv', '/content/drive/MyDrive/Colab_Notebooks/weatherAUS.csv',
])
PRE_PATH = discover([
    'WeatherAUS_preprocessed_MICEcyclic.csv', 'data/WeatherAUS_preprocessed_MICEcyclic.csv',
    '../data/WeatherAUS_preprocessed_MICEcyclic.csv',
    '/content/WeatherAUS_preprocessed_MICEcyclic.csv',
    '/content/drive/MyDrive/Colab_Notebooks/WeatherAUS_preprocessed_MICEcyclic.csv',
])
if RAW_PATH is None or PRE_PATH is None:
    raise FileNotFoundError(f'raw={RAW_PATH}, preprocessed={PRE_PATH}; upload both CSVs or fix paths.')

raw = pd.read_csv(RAW_PATH);  raw['Date'] = pd.to_datetime(raw['Date'])
df  = pd.read_csv(PRE_PATH);  df['Date']  = pd.to_datetime(df['Date'])
df['RainTomorrow'] = df['RainTomorrow'].astype(bool)
print(f'raw         : {raw.shape} from {RAW_PATH}')
print(f'preprocessed: {df.shape} from {PRE_PATH}')

## 3. Missingness profiling

In [ ]:
log_section('3. Missingness profiling (raw, restricted to surviving rows)')

# Original measurement columns whose missingness we audit.
RAW_MEAS_COLS = [
    'MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine','WindGustSpeed',
    'WindSpeed9am','WindSpeed3pm','Humidity9am','Humidity3pm','Pressure9am','Pressure3pm',
    'Cloud9am','Cloud3pm','Temp9am','Temp3pm','WindGustDir','WindDir9am','WindDir3pm','RainToday',
]

# Missingness among only the rows that survived into the modeling frame.
surv = raw.merge(df[['Date','Location']], on=['Date','Location'], how='inner')
miss = (surv[RAW_MEAS_COLS].isna().mean() * 100).sort_values(ascending=False)
print('Raw missing %% among the 138,820 surviving rows:')
print(miss.round(1).to_string())

# Model features (post-engineering) that are heavily imputed -> get indicators.
HEAVY_COLS = [c for c in RAW_MEAS_COLS if miss[c] >= 5.0
              and c in ['Sunshine','Evaporation','Cloud3pm','Cloud9am',
                        'Pressure9am','Pressure3pm','WindGustSpeed']]
KEY_IMPUTED = ['Sunshine','Evaporation','Cloud3pm','Cloud9am']   # the >38%% block
print(f'\nIndicator features will be added for: {HEAVY_COLS}')
print(f'Dose-response key columns           : {KEY_IMPUTED}')

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['crimson' if c in KEY_IMPUTED else ('orange' if c in HEAVY_COLS else 'steelblue')
          for c in miss.index]
ax.barh(miss.index[::-1], miss.values[::-1], color=colors[::-1])
ax.set_xlabel('% imputed (originally missing)')
ax.set_title('Raw missingness among modeling rows\n(red = dose-response key; orange = indicator-added)')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('audit_01_missingness_profile', fig)
plt.show()

## 4. Build imputation mask and indicator features

In [ ]:
log_section('4. Imputation mask + indicators')

mask = raw[['Date','Location']].copy()
for c in RAW_MEAS_COLS:
    mask[f'{c}__imp'] = raw[c].isna().astype(int)
mask = mask.drop_duplicates(['Date','Location'])

n_before = len(df)
df = df.merge(mask, on=['Date','Location'], how='left')
imp_cols = [f'{c}__imp' for c in RAW_MEAS_COLS]
unmatched = df[imp_cols].isna().any(axis=1).sum()
df[imp_cols] = df[imp_cols].fillna(0).astype(int)
print(f'Merged mask onto {n_before:,} rows | unmatched (filled 0): {unmatched:,}')

# Dose-response counter: how many KEY predictors were imputed for each row.
df['n_key_imputed'] = df[[f'{c}__imp' for c in KEY_IMPUTED]].sum(axis=1)
print('\nDistribution of n_key_imputed (0..4):')
print(df['n_key_imputed'].value_counts().sort_index().to_string())

# Imputed share for the heavy columns (for the figure).
share = (df[[f'{c}__imp' for c in HEAVY_COLS]].mean() * 100).round(1)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar([c for c in HEAVY_COLS], [share[f'{c}__imp'] for c in HEAVY_COLS], color='orange')
ax.set_ylabel('% imputed'); ax.set_title('Imputed share of heavy model features (modeling rows)')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
save_fig('audit_02_imputed_share', fig)
plt.show()

## 5. Feature engineering (v2 base + indicators)

In [ ]:
log_section('5. Feature engineering (v2 base + indicators)')

if 'Date_sin' not in df.columns:
    d = df['Date'].dt.dayofyear
    df['Date_sin'] = np.sin(2*np.pi*d/365.25); df['Date_cos'] = np.cos(2*np.pi*d/365.25)

# v2 engineered difference features.
df['Pressure_diff'] = df['Pressure3pm'] - df['Pressure9am']
df['Temp_diff']     = df['Temp3pm']     - df['Temp9am']

DROP_COLS = ['Date','Location','WindGustDir','WindDir9am','WindDir3pm','RainToday',
             'RainTomorrow','Humidity9am','Month','weather_trend_group','Sunshine_trend_group']

BASE_FEATURES = [
    'MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine','WindGustSpeed','WindSpeed9am',
    'WindSpeed3pm','Humidity3pm','Pressure9am','Pressure3pm','Cloud9am','Cloud3pm','Temp9am',
    'Temp3pm','Date_sin','Date_cos','WindGustDir_sin','WindGustDir_cos','WindDir9am_sin',
    'WindDir9am_cos','WindDir3pm_sin','WindDir3pm_cos','Pressure_diff','Temp_diff',
]
IND_FEATURES = [f'{c}__imp' for c in HEAVY_COLS]
PLUS_FEATURES = BASE_FEATURES + IND_FEATURES

missing = [c for c in BASE_FEATURES if c not in df.columns]
assert not missing, f'missing base features: {missing}'
print(f'BASE features : {len(BASE_FEATURES)}')
print(f'+ indicators  : {len(IND_FEATURES)} -> PLUS = {len(PLUS_FEATURES)}')

## 6. Chronological split + config

In [ ]:
log_section('6. Chronological split')

df_sorted = df.sort_values('Date').reset_index(drop=True)
y_sorted = df_sorted['RainTomorrow'].astype(int)
dates = df_sorted['Date']
train_mask = dates < TEST_CUTOFF

X_train = df_sorted.loc[train_mask, PLUS_FEATURES].reset_index(drop=True)
X_test  = df_sorted.loc[~train_mask, PLUS_FEATURES].reset_index(drop=True)
y_train_i = y_sorted[train_mask].reset_index(drop=True)
y_test_i  = y_sorted[~train_mask].reset_index(drop=True)

# imputation counter aligned to the test split (for the dose-response experiment).
nkey_train = df_sorted.loc[train_mask, 'n_key_imputed'].reset_index(drop=True)
nkey_test  = df_sorted.loc[~train_mask, 'n_key_imputed'].reset_index(drop=True)

pos_rate = float(y_train_i.mean())
SPW = (1 - pos_rate) / pos_rate
V2_PARAMS['scale_pos_weight'] = SPW
ts_cv = TimeSeriesSplit(n_splits=5)
print(f'Train {len(X_train):,} | Test {len(X_test):,} | scale_pos_weight {SPW:.4f}')

## 7. Experiment A — do `*_was_imputed` indicators help?

If missingness carries signal (e.g. some stations never report sunshine), adding binary
imputation indicators should lift CV/test F1. A null result means missingness is not directly
informative.

In [ ]:
log_section('7. Experiment A — missingness indicators')

def evaluate(name, cols):
    m = make_xgb(**V2_PARAMS)
    cv_f1 = cross_val_score(m, X_train[cols], y_train_i, cv=ts_cv,
                            scoring='f1', n_jobs=SEARCH_N_JOBS).mean()
    m.fit(X_train[cols], y_train_i)
    proba = m.predict_proba(X_test[cols])[:, 1]
    pred = (proba >= AUDIT_THRESHOLD).astype(int)
    row = dict(model=name, n_feat=len(cols), cv_f1=round(cv_f1, 4),
               test_f1=round(f1_score(y_test_i, pred), 4),
               precision=round(precision_score(y_test_i, pred), 4),
               recall=round(recall_score(y_test_i, pred), 4),
               roc_auc=round(roc_auc_score(y_test_i, proba), 4))
    return m, row, proba

m_base, row_base, proba_base = evaluate('BASE (no indicators)', BASE_FEATURES)
m_plus, row_plus, proba_plus = evaluate('PLUS indicators',      PLUS_FEATURES)

expA = pd.DataFrame([row_base, row_plus])
print(expA.to_string(index=False))
indicator_gain = row_plus['cv_f1'] - row_base['cv_f1']
print(f'\nIndicator CV-F1 gain: {indicator_gain:+.4f}')

In [ ]:
# Gain importance of the indicator features within the PLUS model.
gain = m_plus.get_booster().get_score(importance_type='gain')
ind_gain = pd.DataFrame(
    [(c, gain.get(c, 0.0)) for c in IND_FEATURES],
    columns=['indicator', 'gain']).sort_values('gain', ascending=False)
print('\nIndicator-feature gain (within PLUS model):')
print(ind_gain.to_string(index=False, float_format='%.2f'))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(ind_gain['indicator'][::-1], ind_gain['gain'][::-1], color='orange')
ax.set_xlabel('gain'); ax.set_title('Missingness-indicator importance (PLUS model)')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('audit_03_indicator_importance', fig)
plt.show()

## 8. Experiment B — predictability vs imputation count (dose-response)

Bucket the test rows by how many of the 4 key predictors were imputed (0–4) and score the BASE
model per bucket. **Monotone degradation** of ROC-AUC with imputation count is direct evidence
that imputed rows carry more irreducible error — i.e. imputation noise contributes to the ceiling.
ROC-AUC is the primary (threshold-free) read; F1 is shown for reference.

In [ ]:
log_section('8. Experiment B — dose-response by imputation count')

rows = []
for k in range(5):
    sel = (nkey_test == k).values
    n = int(sel.sum())
    if n < 50 or y_test_i[sel].nunique() < 2:
        rows.append(dict(n_key_imputed=k, n_rows=n, rain_rate=np.nan, roc_auc=np.nan, f1=np.nan))
        continue
    pr = proba_base[sel]
    yt = y_test_i[sel]
    rows.append(dict(
        n_key_imputed=k, n_rows=n, rain_rate=round(float(yt.mean()), 4),
        roc_auc=round(roc_auc_score(yt, pr), 4),
        f1=round(f1_score(yt, (pr >= AUDIT_THRESHOLD).astype(int)), 4)))
doseB = pd.DataFrame(rows)
print(doseB.to_string(index=False))

valid = doseB.dropna(subset=['roc_auc'])
roc_drop = float(valid['roc_auc'].iloc[0] - valid['roc_auc'].iloc[-1])
print(f'\nROC-AUC drop, 0 imputed -> {int(valid["n_key_imputed"].iloc[-1])} imputed: {roc_drop:+.4f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(valid['n_key_imputed'], valid['roc_auc'], 'o-', color='crimson', label='ROC-AUC')
ax.plot(valid['n_key_imputed'], valid['f1'], 's--', color='steelblue', label=f'F1 @ {AUDIT_THRESHOLD}')
ax.set_xlabel('# of key predictors imputed'); ax.set_ylabel('test score')
ax.set_title('Predictability vs imputation count (BASE model)')
ax.set_xticks(range(5)); ax.legend(); ax.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('audit_04_dose_response', fig)
plt.show()

## 9. Experiment C — noise floor: full vs fully-observed

Recompute the EXP-002 k-NN Bayes-error proxy on (i) a full-train subsample and (ii) the
fully-observed subsample (`n_key_imputed == 0`). A **lower** proxy on observed rows means
imputation inflates the local class overlap — the floor is partly an artifact of imputation.

In [ ]:
log_section('9. Experiment C — noise floor (full vs observed-only)')
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

def bayes_proxy(Xdf, y, k=25, n_sub=20000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(Xdf), size=min(n_sub, len(Xdf)), replace=False)
    Xs = StandardScaler().fit_transform(Xdf.iloc[idx].values)
    ys = y.iloc[idx].values
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xs)
    nbr = nn.kneighbors(Xs, return_distance=False)[:, 1:]
    p = ys[nbr].mean(axis=1)
    pred = (p >= 0.5).astype(int)
    return dict(n=len(idx), knn_f1=round(f1_score(ys, pred), 4),
                knn_acc=round((pred == ys).mean(), 4),
                bayes_err=round(float(np.mean(np.minimum(p, 1 - p))), 4))

full = bayes_proxy(X_train[BASE_FEATURES], y_train_i)
obs_sel = (nkey_train == 0).values
clean = bayes_proxy(X_train[BASE_FEATURES][obs_sel].reset_index(drop=True),
                    y_train_i[obs_sel].reset_index(drop=True))
floorC = pd.DataFrame([dict(subset='full', **full), dict(subset='observed-only (0 imputed)', **clean)])
print(floorC.to_string(index=False))
bayes_drop = full['bayes_err'] - clean['bayes_err']
print(f'\nBayes-error proxy drop (full -> observed): {bayes_drop:+.4f}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(floorC['subset'], floorC['bayes_err'], color=['steelblue', 'seagreen'])
ax.set_ylabel('Bayes-error proxy  mean min(p,1-p)')
ax.set_title('Local noise floor: full vs fully-observed rows')
for i, v in enumerate(floorC['bayes_err']):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('audit_05_noise_floor_comparison', fig)
plt.show()

## 10. Verdict

In [ ]:
log_section('10. Verdict')

# Decision thresholds (any one triggering => imputation noise is material).
THR_IND, THR_DOSE, THR_FLOOR = 0.005, 0.030, 0.020
signals = {
    'A: indicator CV-F1 gain >= 0.005': indicator_gain >= THR_IND,
    'B: ROC-AUC drop (0->max imputed) >= 0.030': roc_drop >= THR_DOSE,
    'C: Bayes-proxy drop (full->observed) >= 0.020': bayes_drop >= THR_FLOOR,
}
print('Evidence:')
print(f'  A indicator CV-F1 gain        : {indicator_gain:+.4f}  -> {signals["A: indicator CV-F1 gain >= 0.005"]}')
print(f'  B ROC-AUC dose-response drop  : {roc_drop:+.4f}  -> {signals["B: ROC-AUC drop (0->max imputed) >= 0.030"]}')
print(f'  C Bayes-proxy full->observed  : {bayes_drop:+.4f}  -> {signals["C: Bayes-proxy drop (full->observed) >= 0.020"]}')

n_fire = sum(signals.values())
print(f'\nSignals firing: {n_fire}/3')
if n_fire >= 2:
    print('\nVERDICT: IMPUTATION NOISE is a MATERIAL contributor to the ceiling.')
    print('  -> Keep missingness-indicator features in the model.')
    print('  -> Invest in a causal / better imputer, or restrict to better-observed stations.')
    print('  -> The data-limited ceiling can be partly bought back with data quality.')
elif n_fire == 1:
    print('\nVERDICT: MIXED. One axis suggests imputation noise; treat as a minor lever.')
    print('  -> Add indicators (cheap), but expect the larger gains from NEW inputs (temporal/spatial).')
else:
    print('\nVERDICT: GENUINE AMBIGUITY dominates; imputation is NOT the bottleneck.')
    print('  -> Stop optimizing data quality here.')
    print('  -> Pivot to new signal: temporal/lagged features, spatial context, or a new target.')

## Finalization

In [ ]:
log_section('Finalization')
print(f'Results file: {_RESULT_PATH}')
fig_files = sorted(f for f in os.listdir('figures') if f.startswith('audit_'))
print(f'Audit figures saved ({len(fig_files)}):')
for f in fig_files:
    print(f'  figures/{f}')

sys.stdout = _ORIG_STDOUT
_RESULT_FILE.close()

ZIP_PATH = 'XGB_audit_artifacts.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(_RESULT_PATH)
    for f in fig_files:
        zf.write(f'figures/{f}')
print(f'Bundled artifacts into {ZIP_PATH}')
if IN_COLAB:
    from google.colab import files
    files.download(ZIP_PATH)
print(f'\nDone. See {_RESULT_PATH} and figures/ for outputs.')